# CLIP experts (Pool A)

Trains **one logistic regression per expert** (real vs. that expert's fakes) on the saved CLIP ViT-L/14 features,
the same model as `probe_one_generator.ipynb`. See `clip/train.py`.

- **7 experts**, the same as the ResNet pool (`resnet/experts.py`), on the same `expert_train` images; evaluated on `expert_val`.
- Saved to `models/clip_vit-l14/<expert>.joblib` in the dataset folder, next to the ResNet experts.
- **Safe to rerun:** finished experts are skipped.

In [1]:
import sys
sys.path.insert(0, "../..")  # repo root, so `clip`, `common`, `data` and `resnet` are importable

import pandas as pd

from clip.train import experts_dir, train_all

print("models:", experts_dir())

W1025 14:30:14.587000 62999 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


models: /Users/tim/Developer/SC4001-Metalearners-Route-or-Pool/input/genimage_d8889a9c/models/clip_vit-l14


## Train the 7 experts

In [2]:
infos = train_all()

pd.DataFrame(infos.values()).set_index("expert")[["val_acc", "val_auc", "n_train", "n_val", "train_seconds"]].round(4)

[14:30:22] [ADM] already trained, skipping (delete /Users/tim/Developer/SC4001-Metalearners-Route-or-Pool/input/genimage_d8889a9c/models/clip_vit-l14/ADM.joblib to retrain)
[14:30:22] [BigGAN] already trained, skipping (delete /Users/tim/Developer/SC4001-Metalearners-Route-or-Pool/input/genimage_d8889a9c/models/clip_vit-l14/BigGAN.joblib to retrain)
[14:30:22] [glide] already trained, skipping (delete /Users/tim/Developer/SC4001-Metalearners-Route-or-Pool/input/genimage_d8889a9c/models/clip_vit-l14/glide.joblib to retrain)
[14:30:22] [Midjourney] already trained, skipping (delete /Users/tim/Developer/SC4001-Metalearners-Route-or-Pool/input/genimage_d8889a9c/models/clip_vit-l14/Midjourney.joblib to retrain)
[14:30:22] [SD] already trained, skipping (delete /Users/tim/Developer/SC4001-Metalearners-Route-or-Pool/input/genimage_d8889a9c/models/clip_vit-l14/SD.joblib to retrain)
[14:30:22] [VQDM] already trained, skipping (delete /Users/tim/Developer/SC4001-Metalearners-Route-or-Pool/input/

,val_acc,val_auc,n_train,n_val,train_seconds
expert,,,,,
ADM,0.9590,0.9918,10000,2000,0.3
BigGAN,0.9945,0.9999,10000,2000,0.1
glide,0.9840,0.9988,10000,2000,0.1
Midjourney,0.9565,0.9882,10000,2000,0.3
SD,0.9585,0.9906,10000,2000,0.2
VQDM,0.9585,0.9910,10000,2000,0.3
wukong,0.9415,0.9860,10000,2000,0.2


## Score every image with every expert

Every image of `expert_val`, `meta_train`, `meta_val`, `test` → one raw logit per expert (> 0 = fake), saved as the
feature set `clip_vit-l14_experts`. Columns are in `EXPERTS` order (from `resnet/experts.py`), the same as the ResNet scores. See `clip/score.py`.

In [3]:
from clip.score import SCORES_NAME, score
from data.features import load_features
from resnet.experts import EXPERTS

score()

X, rows = load_features(SCORES_NAME, pool="meta_train")
print(X.shape, list(EXPERTS))

[14:30:25] clip_vit-l14_experts: 96000 images in pools ['expert_val', 'meta_train', 'meta_val', 'test'], 96000 done, 0 to do
(32000, 7) ['ADM', 'BigGAN', 'glide', 'Midjourney', 'SD', 'VQDM', 'wukong']
